In [ ]:
from anthropic import Anthropic
import pandas as pd
import time
import json
import wandb
import os

In [ ]:
client = Anthropic()

In [3]:
df = pd.read_csv('raw/IT_Support_Ticket Data.csv')
print(f'Dataset cargado:\n{len(df)} Tickets\n{len(df.columns)} Columnas')

Dataset cargado:
29651 Tickets
5 Columnas


In [16]:
df.head()

,Unnamed: 0,Body,Department,Priority,Tags
0,0,"Dear Customer Support Team,I am writing to rep...",Technical Support,high,"['Account', 'Disruption', 'Outage', 'IT', 'Tec..."
1,1,"Dear Customer Support Team,I hope this message...",Returns and Exchanges,medium,"['Product', 'Feature', 'Tech Support']"
2,2,"Dear Customer Support Team,I hope this message...",Billing and Payments,low,"['Billing', 'Payment', 'Account', 'Documentati..."
3,3,"Dear Support Team,I hope this message reaches ...",Sales and Pre-Sales,medium,"['Product', 'Feature', 'Feedback', 'Tech Suppo..."
4,4,"Dear Customer Support,I hope this message reac...",Technical Support,high,"['Feature', 'Product', 'Documentation', 'Feedb..."


In [ ]:
df = df.drop(columns=['Unnamed: 0'])

,Body,Department,Priority,Tags
0,"Dear Customer Support Team,I am writing to rep...",Technical Support,high,"['Account', 'Disruption', 'Outage', 'IT', 'Tec..."
1,"Dear Customer Support Team,I hope this message...",Returns and Exchanges,medium,"['Product', 'Feature', 'Tech Support']"
2,"Dear Customer Support Team,I hope this message...",Billing and Payments,low,"['Billing', 'Payment', 'Account', 'Documentati..."
3,"Dear Support Team,I hope this message reaches ...",Sales and Pre-Sales,medium,"['Product', 'Feature', 'Feedback', 'Tech Suppo..."
4,"Dear Customer Support,I hope this message reac...",Technical Support,high,"['Feature', 'Product', 'Documentation', 'Feedb..."
...,...,...,...,...
29646,"Hello Customer Support Team,My name is name. I...",IT Support,high,"['Technical Support', 'Urgent Issue', 'Service..."
29647,"Dear Customer Support Team,We are currently en...",IT Support,high,"['Service Disruption', 'IT Support', 'Urgent I..."
29648,"Dear Customer Support Team,Our Cisco Router IS...",Technical Support,high,"['Technical Support', 'Network Issue', 'Urgent..."
29649,"Dear IT Services Support Team, I am writing to...",Technical Support,high,"['Urgent Issue', 'Service Disruption', 'Incide..."


In [4]:
df['Department'].unique()

<ArrowStringArray>
[              'Technical Support',           'Returns and Exchanges',
            'Billing and Payments',             'Sales and Pre-Sales',
 'Service Outages and Maintenance',                 'Product Support',
                      'IT Support',                'Customer Service',
                 'Human Resources',                 'General Inquiry']
Length: 10, dtype: str

Creamos categorías más genéricas y el equipo encargado de la gestión

In [21]:
CATEGORIES = ['biling', 'technical', 'shipping', 'account', 'general']
URGENCY = ['low', 'medium', 'hgih', 'critical']
TEAMS = ['biling-team', 'tech-support', 'logistics', 'account-team', 'general-support']

In [22]:
#Map del df a nuestras categorías y equipos

DEPARTMENT_TO_CATEGORY = {
    'Technical Support': 'technical',
    'Billing and Payments': 'biling',
    'Service Outages and Maintenance': 'account',
    'IT Support': 'technical',
    'Human Resources': 'general',
    'Returns and Exchanges': 'shipping',
    'Sales and Pre-Sales': 'general',
    'Product Support': 'technical',
    'Customer Service': 'account',
    'General Inquiry': 'general'
}

CATEGORY_TO_TEAM = {
    'technical': 'tech-support',
    'biling': 'biling-team',
    'account': 'account-team',
    'general': 'general-support',
    'shipping': 'logistics'
}

In [26]:
def process_real_tickets(df: pd.DataFrame) -> list[dict]:
    processed = []
    for idx, row in df.iterrows():
        department = str(row.get('Department', 'None'))
        category = DEPARTMENT_TO_CATEGORY.get(department, 'general')
        urgency = str(row.get('Priority', 'None'))
        team = CATEGORY_TO_TEAM.get(category, 'general-support')
        ticket_text = str(row.get('Body', ''))
        if not ticket_text or len(ticket_text) < 5:
            continue
        processed.append({
            'text': ticket_text,
            'category': category,
            'urgency': urgency,
            'team': team
        })
    print(f'Processed {len(processed)} real tickets')
    return processed

In [57]:
def find_underrepresented_cases(dataset: list[dict]) -> list[tuple]:
    min_count = 1500
    counts = {}
    for ticket in dataset:
        key = (ticket['category'], ticket['urgency'])
        counts[key] = counts.get(key, 0) +1
    underrepresented = [key for key, count in counts.items() if count < min_count]
    return underrepresented

In [27]:
tickets_real = process_real_tickets(df)

Processed 29649 real tickets


In [58]:
underrepresented = find_underrepresented_cases(tickets_real)

In [59]:
underrepresented

[('shipping', 'medium'),
 ('biling', 'low'),
 ('general', 'medium'),
 ('general', 'high'),
 ('biling', 'medium'),
 ('shipping', 'high'),
 ('biling', 'high'),
 ('shipping', 'low'),
 ('general', 'low')]

In [ ]:
def generate_synthetic_underrepresented(underrepresented: list[tuple], n_per_case=100) -> list[dict]:
    synthetic = []
    case_descriptions = {
        ('shipping', 'medium'): 'Minor delays, tracking not updating',
        ('biling', 'low'): 'General billing questions, invoice clarification needed',
        ('general', 'medium'): 'General issues with some importance',
        ('general', 'high'): 'Important issues requiring immediate attention',
        ('biling', 'medium'): 'Billing discrepancies, unclear charges, invoice questions',
        ("shipping", "high"): "Package lost, significant delays, major tracking issues",
        ("billing", "high"): "Duplicate charges, unexpected fees, refund requests",
        ("shipping", "low"): "Shipping questions, method preferences",
        ("general", "low"): "General inquiries, feedback, suggestions",
    }
    for category, urgency in underrepresented:
        description = case_descriptions.get((category, urgency), f'{category} issue with {urgency} urgency')
        prompt = f"""Generate {n_per_case} realistic IT support tickets. Return ONLY a JSON array of strings, no other text:
        ["ticket 1", "ticket 2", ...]

        Category: {category}
        Urgency: {urgency}
        Description: {description}
        
        Requirements:
        - Each ticket is a single customer message (1-3 sentences)
        - Vary the tone: frustrated, neutral, polite, urgent
        - Include realistic IT details: software names, error messages, hardware models
        - Some should have typos or grammar issues
        - Make them authentic - real support ticket language
        - Varied length and detail level
        """
        response = client.messages.create(
            model = 'claude-sonnet-4-6',
            max_tokens = 2000,
            messages = [{'role': 'user', 'content': prompt}]
        )
        try:
            tickets_synthetic = json.loads(response.content[0].text)
        except:
            print(f"Error parsing JSON")
            tickets_list = []
        
        for text in tickets_synthetic:
            synthetic.append({
                "text": text,
                "category": category,
                "urgency": urgency,
                "team": CATEGORY_TO_TEAM[category]
            })
        time.sleep(1)
    print(f'Generated {len(synthetic)} synthetic tickets')
    return synthetic        

In [2]:
def save_dataset(dataset: list[dict], filepath: str='processed/dataset.json'):
    os.makedirs(os.path.dirname(filepath), exist_ok=True)

    with open(filepath, 'w') as f:
        json.dump(dataset, f, indent=2, ensure_ascii=False)

    print(f'Dataset saved ti {filepath}')